# Data Leakage Diagnostic — `NUMBER_OF_ISSUES` & related features

This notebook checks whether `NUMBER_OF_ISSUES` (and its derivatives like
`HAS_ISSUE_MTH`) leak future information into the churn prediction target.

**Tests performed:**
1. Univariate distribution of `NUMBER_OF_ISSUES` by target class
2. Cross-tabulation (issue presence vs churn)
3. Point-biserial correlation & mutual information
4. Single-feature ROC-AUC & PR-AUC (a single feature should never get close to the full model)
5. Temporal lag analysis — does the signal concentrate in lag 1 (anchor month)?
6. Feature ablation — full model vs model without suspect features
7. Overlap analysis — what % of churners have issues in the anchor month?

## 0. Configuration

In [ ]:
from datetime import datetime

DATA_CONFIG = {
    "data_path": r"data/churn_targets.parquet",
    "target": "CHURN_2M",
    "drop_cols": [
        "CARD_ID", "CARD_CANCELLATION_DATE", "CHURN_1M", "CHURN_3M",
        "STATUS", "BIRTH_DT", "OPEN_DT", "MONTH_END", "CUST_ID",
        "LAST_EXPOSURE_DATE", "MONTH_YYYYMM",
    ],
    "random_state": 42,
    "test_month_start": datetime(2025, 2, 1),
    "month_window_start": datetime(2022, 2, 1),
    "month_window_end": datetime(2025, 1, 31),
    "majority_reduction_pct": 0.10,
    "n_lags": 12,
    "priority_non_null_cols": [
        "REMAINING_INSTALLMENT_AMOUNT_lag1",
        "REMAINING_INSTALLMENT_COUNT_lag1",
        "TOTAL_EXPOSURE_AMT_1M_lag1",
        "STMT_PYMNT_AMT_lag1",
        "AVG_TICKET_MTH_lag1",
        "MCC_MOST_FREQUENT_MTH_lag1",
    ],
    "use_streaming_collect": True,
}

SUSPECT_PATTERNS = ["NUMBER_OF_ISSUES", "HAS_ISSUE"]

TARGET = DATA_CONFIG["target"]

## 1. Load data

In [ ]:
import os, sys
ROOT = os.path.abspath(".")
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "training"))

from data_processing import load_and_prepare_data

X_train, X_test, y_train, y_test, feature_names, data_info = (
    load_and_prepare_data(DATA_CONFIG)
)

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"Features: {len(feature_names)}")
print(f"Train positive rate: {y_train.mean():.4f}")
print(f"Test  positive rate: {y_test.mean():.4f}")

In [ ]:
import re

suspect_cols = [
    c for c in feature_names
    if any(pat.lower() in c.lower() for pat in SUSPECT_PATTERNS)
]
print(f"Suspect columns ({len(suspect_cols)}):")
for c in sorted(suspect_cols):
    print(f"  {c}")

## 2. Univariate distribution by target class

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

fig, axes = plt.subplots(
    len(suspect_cols), 2,
    figsize=(14, 4 * len(suspect_cols)),
    squeeze=False,
)

for i, col in enumerate(suspect_cols):
    for j, (X, y, label) in enumerate([
        (X_train, y_train, "Train"),
        (X_test, y_test, "Test"),
    ]):
        ax = axes[i, j]
        vals_0 = X.loc[y == 0, col].dropna()
        vals_1 = X.loc[y == 1, col].dropna()
        bins = np.linspace(
            min(vals_0.min(), vals_1.min()),
            max(vals_0.quantile(0.99), vals_1.quantile(0.99)),
            40,
        )
        ax.hist(vals_0, bins=bins, alpha=0.5, label=f"{TARGET}=0", density=True)
        ax.hist(vals_1, bins=bins, alpha=0.5, label=f"{TARGET}=1", density=True)
        ax.set_title(f"{col} — {label}")
        ax.legend()
        ax.set_xlabel(col)
        ax.set_ylabel("Density")

fig.tight_layout()
plt.show()

## 3. Cross-tabulation & overlap analysis

In [ ]:
print("=" * 70)
print("CROSS-TABULATION ANALYSIS (Test set)")
print("=" * 70)

for col in suspect_cols:
    has_value = (X_test[col].fillna(0) > 0).astype(int)
    ct = pd.crosstab(has_value, y_test, margins=True, margins_name="Total")
    ct.index.name = f"{col} > 0"
    ct.columns.name = TARGET
    print(f"\n--- {col} ---")
    print(ct)

    churners_with_issue = ((has_value == 1) & (y_test == 1)).sum()
    total_churners = (y_test == 1).sum()
    pct = 100 * churners_with_issue / max(total_churners, 1)
    print(f"\n  -> {churners_with_issue}/{total_churners} churners ({pct:.1f}%) "
          f"have {col} > 0 at this lag")

    non_churners_with_issue = ((has_value == 1) & (y_test == 0)).sum()
    total_non_churners = (y_test == 0).sum()
    pct_nc = 100 * non_churners_with_issue / max(total_non_churners, 1)
    print(f"  -> {non_churners_with_issue}/{total_non_churners} non-churners ({pct_nc:.1f}%) "
          f"have {col} > 0 at this lag")
    print()

## 4. Point-biserial correlation & mutual information

In [ ]:
from scipy.stats import pointbiserialr
from sklearn.feature_selection import mutual_info_classif

print("=" * 70)
print("CORRELATION & MUTUAL INFORMATION (Test set)")
print("=" * 70)

rows = []
for col in suspect_cols:
    vals = X_test[col].fillna(0).values
    corr, pval = pointbiserialr(y_test, vals)
    mi = mutual_info_classif(
        vals.reshape(-1, 1), y_test, random_state=42, n_neighbors=5
    )[0]
    rows.append({"feature": col, "corr": corr, "p_value": pval, "mutual_info": mi})

corr_df = pd.DataFrame(rows).sort_values("mutual_info", ascending=False)
print(corr_df.to_string(index=False))

## 5. Single-feature predictive power

If a single feature achieves ROC-AUC or PR-AUC close to the full model,
it is almost certainly leaking the target.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("=" * 70)
print("SINGLE-FEATURE PREDICTIVE POWER (Test set)")
print("=" * 70)

rows = []
for col in suspect_cols:
    vals = X_test[col].fillna(0).values
    try:
        roc = roc_auc_score(y_test, vals)
        pr = average_precision_score(y_test, vals)
    except ValueError:
        roc = pr = float("nan")
    rows.append({"feature": col, "ROC_AUC": roc, "PR_AUC": pr})

single_df = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False)
print(single_df.to_string(index=False))

baseline_pr = y_test.mean()
print(f"\nBaseline PR-AUC (positive rate): {baseline_pr:.4f}")
print("If any feature's PR-AUC >> baseline, it alone nearly predicts the target.")

## 6. Temporal lag analysis

If leakage comes from the anchor month, `NUMBER_OF_ISSUES_lag1` should have
dramatically higher predictive power than `_lag2`, `_lag3`, etc.

In [ ]:
import re

lag_pattern = re.compile(r"NUMBER_OF_ISSUES_lag(\d+)")
lag_cols = {}
for c in feature_names:
    m = lag_pattern.match(c)
    if m:
        lag_cols[int(m.group(1))] = c

if lag_cols:
    print("=" * 70)
    print("LAG-BY-LAG PREDICTIVE POWER (NUMBER_OF_ISSUES)")
    print("=" * 70)

    lag_rows = []
    for lag_n in sorted(lag_cols.keys()):
        col = lag_cols[lag_n]
        vals = X_test[col].fillna(0).values
        try:
            roc = roc_auc_score(y_test, vals)
            pr = average_precision_score(y_test, vals)
        except ValueError:
            roc = pr = float("nan")
        frac_nonzero_churn = ((vals > 0) & (y_test == 1)).sum() / max((y_test == 1).sum(), 1)
        frac_nonzero_nochurn = ((vals > 0) & (y_test == 0)).sum() / max((y_test == 0).sum(), 1)
        lag_rows.append({
            "lag": lag_n,
            "feature": col,
            "ROC_AUC": roc,
            "PR_AUC": pr,
            "%_nonzero_churners": round(100 * frac_nonzero_churn, 1),
            "%_nonzero_non_churners": round(100 * frac_nonzero_nochurn, 1),
        })

    lag_df = pd.DataFrame(lag_rows)
    print(lag_df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(lag_df["lag"], lag_df["ROC_AUC"], "o-", color="steelblue")
    axes[0].set_xlabel("Lag")
    axes[0].set_ylabel("ROC-AUC")
    axes[0].set_title("ROC-AUC by lag — NUMBER_OF_ISSUES")
    axes[0].axhline(0.5, color="grey", linestyle="--", alpha=0.5)

    axes[1].plot(lag_df["lag"], lag_df["PR_AUC"], "o-", color="darkorange")
    axes[1].set_xlabel("Lag")
    axes[1].set_ylabel("PR-AUC")
    axes[1].set_title("PR-AUC by lag — NUMBER_OF_ISSUES")
    axes[1].axhline(y_test.mean(), color="grey", linestyle="--", alpha=0.5,
                     label=f"baseline={y_test.mean():.4f}")
    axes[1].legend()

    fig.tight_layout()
    plt.show()

    print("\nIf lag1 has dramatically higher AUC than lag2+, the anchor month is leaking.")
else:
    print("No NUMBER_OF_ISSUES_lag* columns found in features.")

## 7. Feature ablation — model with vs without suspect features

Trains two quick models: one with all features, one without the suspect columns.
A large drop in performance when removing the suspect features confirms over-reliance.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score, recall_score,
)

clean_cols = [c for c in feature_names if c not in suspect_cols]
print(f"Full model features:  {len(feature_names)}")
print(f"Clean model features: {len(clean_cols)}  (removed {len(suspect_cols)})")

hgb_params = dict(
    max_iter=500, max_depth=6, learning_rate=0.05,
    early_stopping=True, random_state=42,
)

print("\nTraining FULL model...")
clf_full = HistGradientBoostingClassifier(**hgb_params)
clf_full.fit(X_train, y_train)
y_prob_full = clf_full.predict_proba(X_test)[:, 1]

print("Training CLEAN model (without suspect features)...")
clf_clean = HistGradientBoostingClassifier(**hgb_params)
clf_clean.fit(X_train[clean_cols], y_train)
y_prob_clean = clf_clean.predict_proba(X_test[clean_cols])[:, 1]

def eval_model(y_true, y_prob, label):
    from sklearn.metrics import precision_recall_curve
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = np.where((prec[:-1] + rec[:-1]) > 0,
                   2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1]), 0)
    best_thr = thr[np.argmax(f1s)]
    y_pred = (y_prob >= best_thr).astype(int)
    return {
        "model": label,
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "threshold": best_thr,
    }

results = pd.DataFrame([
    eval_model(y_test, y_prob_full, "FULL (all features)"),
    eval_model(y_test, y_prob_clean, f"CLEAN (no {'/'.join(SUSPECT_PATTERNS)})"),
])

print("\n" + "=" * 70)
print("FEATURE ABLATION COMPARISON")
print("=" * 70)
print(results.to_string(index=False))

delta_pr = results.iloc[0]["PR_AUC"] - results.iloc[1]["PR_AUC"]
delta_roc = results.iloc[0]["ROC_AUC"] - results.iloc[1]["ROC_AUC"]
print(f"\nDelta PR-AUC:  {delta_pr:+.4f}")
print(f"Delta ROC-AUC: {delta_roc:+.4f}")

if delta_pr > 0.05:
    print("\n⚠ LARGE DROP — suspect features carry outsized predictive power.")
    print("  This strongly suggests target leakage.")
elif delta_pr > 0.02:
    print("\n⚠ MODERATE DROP — further investigation recommended.")
else:
    print("\nSmall or no drop — suspect features are not driving the model.")

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

PrecisionRecallDisplay.from_predictions(y_test, y_prob_full, ax=axes[0],
                                         name="Full model")
PrecisionRecallDisplay.from_predictions(y_test, y_prob_clean, ax=axes[0],
                                         name="Clean model")
axes[0].set_title("Precision-Recall Curve")

RocCurveDisplay.from_predictions(y_test, y_prob_full, ax=axes[1],
                                  name="Full model")
RocCurveDisplay.from_predictions(y_test, y_prob_clean, ax=axes[1],
                                  name="Clean model")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_title("ROC Curve")

fig.tight_layout()
plt.show()

## 8. Permutation importance — CLEAN model

Run permutation importance on the clean model to see the new top features.

In [ ]:
from sklearn.inspection import permutation_importance

sample_idx = X_test[clean_cols].sample(
    n=min(2000, len(X_test)), random_state=42
).index

perm = permutation_importance(
    clf_clean,
    X_test.loc[sample_idx, clean_cols],
    y_test[sample_idx],
    n_repeats=3,
    random_state=42,
    scoring="average_precision",
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame({
        "feature": clean_cols,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("Top 25 features (CLEAN model, no suspect features):")
print(perm_df.head(25).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
plot_df = perm_df.head(20).iloc[::-1]
ax.barh(plot_df["feature"], plot_df["importance_mean"],
        xerr=plot_df["importance_std"])
ax.set_xlabel("Mean decrease in PR-AUC (permutation)")
ax.set_title("Top 20 Features — Permutation Importance (CLEAN model)")
fig.tight_layout()
plt.show()

## 9. Summary & recommendation

In [ ]:
print("=" * 70)
print("LEAKAGE DIAGNOSTIC SUMMARY")
print("=" * 70)

print(f"""
Suspect columns removed: {suspect_cols}

RESULTS:
  Full model   PR-AUC: {results.iloc[0]['PR_AUC']:.4f}   ROC-AUC: {results.iloc[0]['ROC_AUC']:.4f}
  Clean model  PR-AUC: {results.iloc[1]['PR_AUC']:.4f}   ROC-AUC: {results.iloc[1]['ROC_AUC']:.4f}
  Delta        PR-AUC: {delta_pr:+.4f}          ROC-AUC: {delta_roc:+.4f}
""")

print("LEAKAGE MECHANISM:")
print("""
  The SQL query (06_tlmk_monthly.sql) counts issues where:
    TRUNC(iss.ISSUE_CREATEDATE, 'MM') = TRUNC(bc.MONTH_END, 'MM')

  This means NUMBER_OF_ISSUES at month T includes issues filed IN month T.
  When a customer requests cancellation, an issue is created as part of
  (or immediately before) the cancellation process. So:

    Month T  : Customer calls to cancel -> issue created -> NUMBER_OF_ISSUES counts it
    Month T+1: Card officially cancelled -> CARD_CANCELLATION_DATE set

  The model's anchor month (lag 1) sees the issue that IS the churn event.
  This is textbook target leakage.
""")

print("RECOMMENDATION:")
print("""
  Option A: Drop NUMBER_OF_ISSUES and HAS_ISSUE_MTH entirely from the model.
            Add them to drop_cols in your DATA_CONFIG.

  Option B: Lag the issue count by at least 1 extra month in the SQL query:
            COUNT issues where ISSUE_CREATEDATE < TRUNC(MONTH_END, 'MM')
            This ensures you only see issues from BEFORE the observation month.

  Option C: Filter out cancellation-related issue types from the COUNT,
            keeping only complaint/inquiry types that precede churn behavior.
""")